# Tadhkirat — Plant Identification & Translation Pipeline v1

## What this does:
1. Loads `plants_only_v2.csv` (649 confirmed plants)
2. For each plant:
   - Feeds Arabic names + full paragraph to Gemini 3 Flash with Google Search grounding
   - Identifies what plant this actually is (not just translation — contextual identification)
   - Finds all known names across languages
   - Validates scientific name against GBIF botanical database
3. Saves to `plant_identification_v1.csv` after every entry
4. Exports final Excel file

## Output columns:
- entry_id
- arabic_names (primary + all synonyms)
- common_english_name (blank if none exists)
- other_known_names (other languages/names)
- scientific_name
- scientific_name_confidence
- gbif_validated
- translation_note
- model_accuracy
- needs_human_review

## Cell 1 — Mount Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── PATHS ───────────────────────────────────────────────────────
DRIVE_FOLDER   = '/content/drive/MyDrive/tadhkirat_dawood'
PLANTS_CSV     = f'{DRIVE_FOLDER}/plants_only_v2.csv'
OUTPUT_CSV     = f'{DRIVE_FOLDER}/plant_identification_v1.csv'
OUTPUT_EXCEL   = f'{DRIVE_FOLDER}/plant_identification_v1.xlsx'
PROGRESS_FILE  = f'{DRIVE_FOLDER}/identification_progress_v1.txt'

# ── VERTEX AI ───────────────────────────────────────────────────
PROJECT_ID = 'project-7f940e6f-ba1d-404b-948'
LOCATION   = 'global'
MODEL_ID   = 'gemini-3-flash-preview'

print(f'Plants CSV : {"✓ Found" if os.path.exists(PLANTS_CSV) else "✗ NOT FOUND"}')
print(f'Output CSV : {OUTPUT_CSV}')

## Cell 2 — Install Dependencies

In [ ]:
!pip install google-genai pandas openpyxl requests -q
print('✓ Dependencies installed.')

## Cell 3 — Authenticate & Initialize Gemini

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

# Test
test = client.models.generate_content(
    model=MODEL_ID,
    contents='قل نعم فقط',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )
)
print(f'✓ Gemini 3 Flash ready: {test.text.strip()[:50]}')

## Cell 4 — GBIF Validation Helper

In [ ]:
import requests

def validate_with_gbif(scientific_name: str) -> dict:
    """
    Validates a scientific name against the GBIF botanical database.
    GBIF is the Global Biodiversity Information Facility.
    Completely free, no API key needed.

    Returns:
        validated      : True/False
        accepted_name  : The canonical accepted scientific name
        confidence     : GBIF match confidence 0-100
        rank           : SPECIES, GENUS, FAMILY etc.
    """
    if not scientific_name or scientific_name.strip() in ['', 'Unknown', 'N/A']:
        return {
            'validated': False,
            'accepted_name': '',
            'confidence': 0,
            'rank': ''
        }

    try:
        url = 'https://api.gbif.org/v1/species/match'
        params = {
            'name': scientific_name.strip(),
            'kingdom': 'Plantae',  # restrict to plants only
            'verbose': False
        }
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        match_type   = data.get('matchType', 'NONE')
        confidence   = data.get('confidence', 0)
        accepted_name = data.get('species', '') or data.get('genus', '') or ''
        rank         = data.get('rank', '')
        status       = data.get('status', '')

        validated = (
            match_type in ['EXACT', 'FUZZY'] and
            confidence >= 80
        )

        # Use accepted name if available
        if data.get('acceptedUsageKey'):
            accepted_name = data.get('species', accepted_name)

        return {
            'validated':     validated,
            'accepted_name': accepted_name,
            'confidence':    confidence,
            'rank':          rank
        }

    except Exception as e:
        return {
            'validated': False,
            'accepted_name': '',
            'confidence': 0,
            'rank': ''
        }

# Test GBIF
test_gbif = validate_with_gbif('Coriandrum sativum')
print(f'✓ GBIF ready. Test (Coriandrum sativum):')
print(f'  Validated     : {test_gbif["validated"]}')
print(f'  Accepted name : {test_gbif["accepted_name"]}')
print(f'  Confidence    : {test_gbif["confidence"]}')

## Cell 5 — Identification Prompt Builder

This prompt asks Gemini to:
1. Search Google for the plant using all its Arabic names as context
2. Identify what plant this actually IS — not just translate the name
3. Find all known names across languages
4. Return a structured response

In [ ]:
def build_identification_prompt(
    primary_name: str,
    synonyms: str,
    paragraph: str
) -> str:
    """
    Builds the plant identification prompt.
    Instructs Gemini to search Google and identify the plant
    from its classical Arabic description.
    """
    # Build the names section
    all_names = primary_name
    if synonyms and str(synonyms).strip() not in ['', 'nan']:
        all_names += f' (also known as: {synonyms})'

    prompt = f"""أنت خبير في النباتات الطبية وتاريخ الصيدلة الإسلامية.

لديك مدخل من كتاب "تذكرة أولى الألباب" لداود الأنطاكي (القرن السادس عشر الميلادي).
هذا النبات موثق ومعروف في كتب الطب الإسلامي التقليدي.

الاسم العربي: {all_names}

الوصف من الكتاب:
{paragraph[:600]}

المطلوب:
١. ابحث على الإنترنت عن هذا النبات باستخدام اسمه العربي ووصفه لتحديد هويته العلمية
٢. حدد الاسم العلمي الصحيح (Scientific Name) بناءً على الوصف النباتي
٣. ابحث عن كل أسماء هذا النبات المعروفة بالإنجليزية والعربية وغيرها
٤. إذا كان للنبات اسم إنجليزي شائع فاذكره، وإذا لم يكن له اسم إنجليزي شائع اتركه فارغاً
٥. اذكر درجة ثقتك في التحديد

IMPORTANT RULES:
- Base identification on the DESCRIPTION in the text, not just the name
- If the Arabic name is a direct description (like 'crow's foot'), find what plant matches that description AND name
- Include Persian, Turkish, Syriac names if known and relevant
- If no English common name exists, write exactly: NO_ENGLISH_NAME
- If scientific name is uncertain, write the most likely one with low confidence
- Never guess a scientific name you are not reasonably sure about

Respond ONLY in this exact format, no extra text:
COMMON_ENGLISH_NAME: [common English name OR NO_ENGLISH_NAME]
SCIENTIFIC_NAME: [Genus species OR Unknown]
OTHER_NAMES: [comma separated list of other known names in any language]
SCIENTIFIC_CONFIDENCE: [0.0 to 1.0]
TRANSLATION_NOTE: [one sentence explaining how you identified this plant]
MODEL_ACCURACY: [0.0 to 1.0 overall confidence in the identification]"""

    return prompt

print('✓ Prompt builder ready.')

## Cell 6 — Identification Function

In [ ]:
import time
import re

def identify_plant(row: dict) -> dict:
    """
    Full identification pipeline for one plant entry.
    1. Builds prompt with all Arabic names + paragraph
    2. Calls Gemini with Google Search grounding
    3. Parses structured response
    4. Validates scientific name with GBIF
    5. Returns complete result dict
    """
    primary_name = str(row['primary_arabic_name'])
    synonyms     = str(row.get('synonyms', ''))
    paragraph    = str(row['full_paragraph'])
    entry_id     = row['entry_id']

    # Build all Arabic names combined
    if synonyms and synonyms not in ['', 'nan']:
        arabic_names = f"{primary_name} | {synonyms}"
    else:
        arabic_names = primary_name

    # ── Step 1: Gemini identification with Google Search ──────
    prompt = build_identification_prompt(primary_name, synonyms, paragraph)

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())]
            )
        )
        text = response.text.strip()

        # ── Parse response ────────────────────────────────────
        common_english_name  = ''
        scientific_name      = 'Unknown'
        other_names          = ''
        scientific_confidence = 0.5
        translation_note     = ''
        model_accuracy       = 0.5

        for line in text.split('\n'):
            line = line.strip()

            if line.upper().startswith('COMMON_ENGLISH_NAME:'):
                val = line.split(':', 1)[1].strip()
                if val == 'NO_ENGLISH_NAME' or not val:
                    common_english_name = ''
                else:
                    common_english_name = val

            elif line.upper().startswith('SCIENTIFIC_NAME:'):
                val = line.split(':', 1)[1].strip()
                scientific_name = val if val and val != 'Unknown' else 'Unknown'

            elif line.upper().startswith('OTHER_NAMES:'):
                other_names = line.split(':', 1)[1].strip()

            elif line.upper().startswith('SCIENTIFIC_CONFIDENCE:'):
                try:
                    nums = re.findall(r'[0-9.]+', line.split(':', 1)[1])
                    if nums:
                        scientific_confidence = float(nums[0])
                        scientific_confidence = min(max(scientific_confidence, 0.0), 1.0)
                except:
                    pass

            elif line.upper().startswith('TRANSLATION_NOTE:'):
                translation_note = line.split(':', 1)[1].strip()

            elif line.upper().startswith('MODEL_ACCURACY:'):
                try:
                    nums = re.findall(r'[0-9.]+', line.split(':', 1)[1])
                    if nums:
                        model_accuracy = float(nums[0])
                        model_accuracy = min(max(model_accuracy, 0.0), 1.0)
                except:
                    pass

    except Exception as e:
        return {
            'entry_id':              entry_id,
            'arabic_names':          arabic_names,
            'common_english_name':   '',
            'other_known_names':     '',
            'scientific_name':       'Unknown',
            'scientific_confidence': 0.0,
            'gbif_validated':        False,
            'gbif_accepted_name':    '',
            'translation_note':      f'Error: {str(e)[:100]}',
            'model_accuracy':        0.0,
            'needs_human_review':    'Yes'
        }

    # ── Step 2: Validate with GBIF ────────────────────────────
    gbif_result = {'validated': False, 'accepted_name': '', 'confidence': 0}

    if scientific_name and scientific_name != 'Unknown':
        gbif_result = validate_with_gbif(scientific_name)

        # If GBIF found a better accepted name, use it
        if gbif_result['validated'] and gbif_result['accepted_name']:
            scientific_name = gbif_result['accepted_name']

    # ── Step 3: Determine review flag ─────────────────────────
    needs_review = (
        model_accuracy < 0.70 or
        scientific_name == 'Unknown' or
        not gbif_result['validated']
    )

    return {
        'entry_id':              entry_id,
        'arabic_names':          arabic_names,
        'common_english_name':   common_english_name,
        'other_known_names':     other_names,
        'scientific_name':       scientific_name,
        'scientific_confidence': scientific_confidence,
        'gbif_validated':        gbif_result['validated'],
        'gbif_accepted_name':    gbif_result.get('accepted_name', ''),
        'translation_note':      translation_note,
        'model_accuracy':        model_accuracy,
        'needs_human_review':    'Yes' if needs_review else 'No'
    }

print('✓ Identification function ready.')

## Cell 7 — CSV Save/Load Helpers

In [ ]:
import pandas as pd

OUTPUT_COLUMNS = [
    'entry_id',
    'arabic_names',
    'common_english_name',
    'other_known_names',
    'scientific_name',
    'scientific_confidence',
    'gbif_validated',
    'gbif_accepted_name',
    'translation_note',
    'model_accuracy',
    'needs_human_review'
]

def load_output_csv() -> pd.DataFrame:
    if os.path.exists(OUTPUT_CSV):
        return pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
    return pd.DataFrame(columns=OUTPUT_COLUMNS)

def save_row(result: dict):
    df  = load_output_csv()
    df  = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

def save_progress(idx: int):
    with open(PROGRESS_FILE, 'w') as f:
        f.write(str(idx))

def load_progress() -> int:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return int(f.read().strip())
    return -1

print('✓ CSV helpers ready.')

## Cell 8 — Load Plants & Test on First Entry
Run this to verify everything works before the full loop.

In [ ]:
# Load plants
df_plants = pd.read_csv(PLANTS_CSV, encoding='utf-8-sig')
print(f'✓ Loaded {len(df_plants)} plants from plants_only_v2.csv')
print(f'  Columns: {list(df_plants.columns)}')
print()

# Test on first entry
print('Testing on first entry...')
test_row = df_plants.iloc[0].to_dict()
print(f'  Plant: [{test_row["primary_arabic_name"]}]')

test_result = identify_plant(test_row)

print()
print('=== TEST RESULT ===')
for k, v in test_result.items():
    print(f'  {k}: {v}')

## Cell 9 — Main Identification Loop

Processes all 649 plants.
Saves after every single entry.
Validates each scientific name against GBIF.
Fully resumable — safe to disconnect anytime.

In [ ]:
last_done  = load_progress()
resume_idx = last_done + 1
total      = len(df_plants)

already_done = len(load_output_csv())

print(f'Total plants          : {total}')
print(f'Already identified    : {already_done}')
print(f'Remaining             : {total - resume_idx}')
print(f'Output                : {OUTPUT_CSV}')
print('=' * 60)

gbif_validated_count   = 0
has_english_count      = 0
needs_review_count     = 0

for idx in range(resume_idx, total):
    row = df_plants.iloc[idx].to_dict()

    # Identify the plant
    result = identify_plant(row)

    # Save immediately
    save_row(result)
    save_progress(idx)

    # Track stats
    if result['gbif_validated']:    gbif_validated_count += 1
    if result['common_english_name']: has_english_count  += 1
    if result['needs_human_review'] == 'Yes': needs_review_count += 1

    # Progress print
    gbif_icon    = '✓' if result['gbif_validated'] else '✗'
    english_icon = '🌿' if result['common_english_name'] else '—'

    print(
        f'[{idx+1:>3}/{total}] {english_icon} '
        f'[{result["arabic_names"][:30]}] '
        f'→ {result["scientific_name"][:35]} '
        f'GBIF:{gbif_icon} '
        f'ACC:{result["model_accuracy"]:.2f}'
    )

    # Rate limit
    time.sleep(1.5)

print('\n' + '=' * 60)
print('IDENTIFICATION COMPLETE')
print(f'Total identified      : {total}')
print(f'GBIF validated        : {gbif_validated_count}')
print(f'Has English name      : {has_english_count}')
print(f'Needs human review    : {needs_review_count}')
print(f'Output                : {OUTPUT_CSV}')

## Cell 10 — Final Summary & Excel Export

In [ ]:
df_out = load_output_csv()

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'Total entries            : {len(df_out)}')
print(f'GBIF validated           : {df_out["gbif_validated"].sum()}')
print(f'Has English name         : {(df_out["common_english_name"] != "").sum()}')
print(f'No English name          : {(df_out["common_english_name"] == "").sum()}')
print(f'Scientific name found    : {(df_out["scientific_name"] != "Unknown").sum()}')
print(f'Unknown scientific name  : {(df_out["scientific_name"] == "Unknown").sum()}')
print(f'High accuracy (≥0.85)    : {(df_out["model_accuracy"] >= 0.85).sum()}')
print(f'Needs human review       : {(df_out["needs_human_review"] == "Yes").sum()}')
print(f'Has synonyms             : {df_out["arabic_names"].str.contains("|", regex=False).sum()}')
print()

# Export Excel with multiple sheets
with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:

    # Sheet 1 — All plants
    df_out.to_excel(writer, sheet_name='All_Plants', index=False)

    # Sheet 2 — GBIF validated only
    df_out[df_out['gbif_validated'] == True].to_excel(
        writer, sheet_name='GBIF_Validated', index=False
    )

    # Sheet 3 — Has English name
    df_out[df_out['common_english_name'] != ''].to_excel(
        writer, sheet_name='Has_English_Name', index=False
    )

    # Sheet 4 — No English name (obscure plants)
    df_out[df_out['common_english_name'] == ''].to_excel(
        writer, sheet_name='No_English_Name', index=False
    )

    # Sheet 5 — Needs human review
    df_out[df_out['needs_human_review'] == 'Yes'].to_excel(
        writer, sheet_name='Needs_Review', index=False
    )

    # Sheet 6 — High confidence (model_accuracy >= 0.85)
    df_out[df_out['model_accuracy'] >= 0.85].to_excel(
        writer, sheet_name='High_Confidence', index=False
    )

print(f'✓ Excel saved: {OUTPUT_EXCEL}')
print(f'  Sheet 1 All_Plants       : {len(df_out)} rows')
print(f'  Sheet 2 GBIF_Validated   : {df_out["gbif_validated"].sum()} rows')
print(f'  Sheet 3 Has_English_Name : {(df_out["common_english_name"] != "").sum()} rows')
print(f'  Sheet 4 No_English_Name  : {(df_out["common_english_name"] == "").sum()} rows')
print(f'  Sheet 5 Needs_Review     : {(df_out["needs_human_review"] == "Yes").sum()} rows')
print(f'  Sheet 6 High_Confidence  : {(df_out["model_accuracy"] >= 0.85).sum()} rows')